In [1]:
import pandas as pd

catboost_parquet_path = "s3://aiml-final/catboost/catboost_sample_1M"

pdf = pd.read_parquet(catboost_parquet_path)

print("Pandas DataFrame shape:", pdf.shape)
print(pdf.head())

Pandas DataFrame shape: (999856, 24)
                     id  click      hour    C1 banner_pos   site_id  \
0  11264398686579609867      1  14102206  1005          0  1fbe01fe   
1  11344561052336927187      0  14102206  1005          0  85f751fd   
2  12644600666206918062      0  14102206  1005          1  a7853007   
3  13141136602608368548      0  14102206  1005          0  e149eb68   
4  13452737095920931736      0  14102206  1005          0  1fbe01fe   

  site_domain site_category    app_id app_domain  ... device_type  \
0    f3845767      28905ebd  ecad2386   7801e8d9  ...           1   
1    c4e18dd6      50e219e0  7358e05e   b9528b13  ...           1   
2    7e091613      f028772b  ecad2386   7801e8d9  ...           1   
3    91cdcccd      f028772b  ecad2386   7801e8d9  ...           1   
4    f3845767      28905ebd  ecad2386   7801e8d9  ...           1   

  device_conn_type    C14  C15 C16   C17 C18  C19     C20  C21  
0                0  21726  320  50  2502   0   35      -

In [2]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss

y = pdf["click"].astype(int)

X = pdf.drop(columns=["click", "id"])

if "hour" in X.columns:
    X["hour_of_day"] = X["hour"].astype(str).str[-2:].astype(int)

print("X shape:", X.shape)
print("y positive rate:", y.mean())

# baseline
pos_rate = y.mean()
baseline_pred = np.full(len(y), pos_rate, dtype=float)
baseline_logloss = log_loss(y, baseline_pred)

print(f"Baseline: p={pos_rate:.4f}, LogLoss={baseline_logloss:.4f}")


X shape: (999856, 23)
y positive rate: 0.17036953321278264
Baseline: p=0.1704, LogLoss=0.4565


In [11]:
# find all categorical cols
cat_cols = [c for c in X.columns if c != "hour_of_day"]
cat_features = [X.columns.get_loc(c) for c in cat_cols]

print("Num categorical features:", len(cat_features))
print("Example categorical cols:", cat_cols[:10])

# split train / val，keep 0/1 proportion
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=2025,
    stratify=y
)

print("Train shape:", X_train.shape, " Val shape:", X_val.shape)

# count weight
neg_cnt = (y_train == 0).sum()
pos_cnt = (y_train == 1).sum()
class_weights = [1.0, neg_cnt / pos_cnt]

print("Class weights (0,1):", class_weights)


Num categorical features: 22
Example categorical cols: ['hour', 'C1', 'banner_pos', 'site_id', 'site_domain', 'site_category', 'app_id', 'app_domain', 'app_category', 'device_id']
Train shape: (799884, 23)  Val shape: (199972, 23)
Class weights (0,1): [1.0, 4.869588188675922]


In [14]:
from catboost import CatBoostClassifier, Pool

def train_and_eval_catboost(X_train, y_train, X_val, y_val,
                            cat_features, params):
    train_pool = Pool(X_train, y_train, cat_features=cat_features)
    val_pool   = Pool(X_val,   y_val,   cat_features=cat_features)

    model = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="Logloss",
        depth=params["depth"],
        learning_rate=params["learning_rate"],
        l2_leaf_reg=params["L2_leaf_reg"],
        iterations=3000,
        od_type="Iter",
        od_wait=100,
        random_seed=2025,
        verbose=100,
        thread_count=1,
    )

    print(">>> Train with params:", params)
    model.fit(train_pool, eval_set=val_pool)

    from sklearn.metrics import log_loss
    val_pred = model.predict_proba(val_pool)[:, 1]
    val_ll = log_loss(y_val, val_pred)
    print(f"Val logloss = {val_ll:.6f}, best iteration = {model.get_best_iteration()}")

    return model, val_ll


In [ ]:
param_grid = [
    # 1. Around the original baseline: depth 6, lr 0.05, L2 3
    {"depth": 6, "learning_rate": 0.05, "L2_leaf_reg": 3},

    # 2. Same depth, smaller learning rate (more conservative)
    {"depth": 6, "learning_rate": 0.03, "L2_leaf_reg": 3},

    # 3. Slightly deeper tree to see if more capacity helps
    {"depth": 7, "learning_rate": 0.05, "L2_leaf_reg": 3},

    # 4. Deeper + smaller lr to reduce potential overfitting
    {"depth": 7, "learning_rate": 0.03, "L2_leaf_reg": 3},

    # 5. Stronger L2 regularization to see if logloss gets smoother
    {"depth": 6, "learning_rate": 0.05, "L2_leaf_reg": 5},
]


results = []
best_logloss = float("inf")
best_model = None
best_params = None

for i, p in enumerate(param_grid, 1):
    print(f"\n========== {i}/{len(param_grid)} ==========")
    model, val_ll = train_and_eval_catboost(
        X_train, y_train, X_val, y_val,
        cat_features=cat_features,
        params=p,
    )

    results.append({
        "params": p,
        "val_logloss": val_ll,
        "best_iteration": model.get_best_iteration(),
    })

    if val_ll < best_logloss:
        best_logloss = val_ll
        best_model = model
        best_params = p

print("\n===== Grid Search Finished =====")
print("Best params:", best_params)
print(f"Best val logloss: {best_logloss:.6f}")



========== 1/5 ==========
>>> Train with params: {'depth': 6, 'learning_rate': 0.05, 'L2_leaf_reg': 3}
0:	learn: 0.6609281	test: 0.6605880	best: 0.6605880 (0)	total: 3.88s	remaining: 2h 9m 13s
100:	learn: 0.4009268	test: 0.3988382	best: 0.3988382 (100)	total: 7m 42s	remaining: 2h 25m 2s
200:	learn: 0.3986273	test: 0.3963926	best: 0.3963926 (200)	total: 16m 28s	remaining: 2h 27m 24s
300:	learn: 0.3975421	test: 0.3954497	best: 0.3954497 (300)	total: 25m 7s	remaining: 2h 21m 48s
400:	learn: 0.3967775	test: 0.3948684	best: 0.3948684 (400)	total: 33m 30s	remaining: 2h 13m 36s
500:	learn: 0.3962023	test: 0.3944829	best: 0.3944829 (500)	total: 42m 45s	remaining: 2h 7m 55s
600:	learn: 0.3957913	test: 0.3942883	best: 0.3942882 (599)	total: 51m 38s	remaining: 2h 12s
700:	learn: 0.3953833	test: 0.3940889	best: 0.3940889 (700)	total: 1h 30s	remaining: 1h 52m 7s
800:	learn: 0.3950585	test: 0.3939318	best: 0.3939318 (800)	total: 1h 9m 18s	remaining: 1h 43m 45s
900:	learn: 0.3947224	test: 0.3937968	

In [15]:
best_params = {'depth': 6, 'learning_rate': 0.05, 'L2_leaf_reg': 3}

best_model, best_ll = train_and_eval_catboost(
    X_train, y_train, X_val, y_val,
    cat_features=cat_features,
    params=best_params
)

print("best_ll:", best_ll)


>>> Train with params: {'depth': 6, 'learning_rate': 0.05, 'L2_leaf_reg': 3}
0:	learn: 0.6609281	test: 0.6605880	best: 0.6605880 (0)	total: 4.12s	remaining: 2h 17m 25s
100:	learn: 0.4009268	test: 0.3988382	best: 0.3988382 (100)	total: 7m 55s	remaining: 2h 29m 1s
200:	learn: 0.3986273	test: 0.3963926	best: 0.3963926 (200)	total: 16m 58s	remaining: 2h 31m 57s
300:	learn: 0.3975421	test: 0.3954497	best: 0.3954497 (300)	total: 26m 3s	remaining: 2h 27m 4s
400:	learn: 0.3967775	test: 0.3948684	best: 0.3948684 (400)	total: 34m 44s	remaining: 2h 18m 33s
500:	learn: 0.3962023	test: 0.3944829	best: 0.3944829 (500)	total: 43m 52s	remaining: 2h 11m 17s
600:	learn: 0.3957913	test: 0.3942883	best: 0.3942882 (599)	total: 52m 50s	remaining: 2h 3m 1s
700:	learn: 0.3953833	test: 0.3940889	best: 0.3940889 (700)	total: 1h 2m	remaining: 1h 54m 53s
800:	learn: 0.3950585	test: 0.3939318	best: 0.3939318 (800)	total: 1h 11m 5s	remaining: 1h 46m 24s
900:	learn: 0.3947224	test: 0.3937968	best: 0.3937968 (900)	to

In [16]:
print("train cols:", X_train.shape[1], "cat_features:", len(cat_features))

train cols: 23 cat_features: 22


In [17]:
from catboost import Pool
import numpy as np
import pandas as pd

S3_BUCKET = "s3://aiml-final"
TEST_PATH   = f"{S3_BUCKET}/ProjectTestData.csv"
SUBMIT_PATH = f"{S3_BUCKET}/ProjectSubmission-TeamX.csv"
OUT_PATH    = f"{S3_BUCKET}/ProjectSubmission-TeamX-FILLED.csv"

sub_df = pd.read_csv(SUBMIT_PATH, storage_options={"anon": False})
assert "P(click)" in sub_df.columns

CHUNK_SIZE = 500_000
preds = np.empty(len(sub_df), dtype=np.float32)

start = 0
reader = pd.read_csv(TEST_PATH, chunksize=CHUNK_SIZE, storage_options={"anon": False})

for k, chunk in enumerate(reader, 1):
    if "hour" in chunk.columns:
        chunk["hour_of_day"] = chunk["hour"].astype(str).str[-2:].astype(int)

    X_chunk = chunk.drop(columns=["id"])
    X_chunk = X_chunk[X_train.columns]
    assert list(X_chunk.columns) == list(X_train.columns)

    pool = Pool(X_chunk, cat_features=cat_features)
    proba = best_model.predict_proba(pool)[:, 1].astype(np.float32)

    end = start + len(proba)
    preds[start:end] = proba
    print(f"Chunk {k}: wrote preds [{start}:{end}]")
    start = end

assert start == len(sub_df), (start, len(sub_df))

sub_df["P(click)"] = preds
sub_df.to_csv(OUT_PATH, index=False, storage_options={"anon": False})

print("Saved to:", OUT_PATH)
print(sub_df.head())
print("P(click) range:", float(preds.min()), float(preds.max()))


Chunk 1: wrote preds [0:500000]
Chunk 2: wrote preds [500000:1000000]
Chunk 3: wrote preds [1000000:1500000]
Chunk 4: wrote preds [1500000:2000000]
Chunk 5: wrote preds [2000000:2500000]
Chunk 6: wrote preds [2500000:3000000]
Chunk 7: wrote preds [3000000:3500000]
Chunk 8: wrote preds [3500000:4000000]
Chunk 9: wrote preds [4000000:4500000]
Chunk 10: wrote preds [4500000:5000000]
Chunk 11: wrote preds [5000000:5500000]
Chunk 12: wrote preds [5500000:6000000]
Chunk 13: wrote preds [6000000:6500000]
Chunk 14: wrote preds [6500000:7000000]
Chunk 15: wrote preds [7000000:7500000]
Chunk 16: wrote preds [7500000:8000000]
Chunk 17: wrote preds [8000000:8500000]
Chunk 18: wrote preds [8500000:9000000]
Chunk 19: wrote preds [9000000:9500000]
Chunk 20: wrote preds [9500000:10000000]
Chunk 21: wrote preds [10000000:10500000]
Chunk 22: wrote preds [10500000:11000000]
Chunk 23: wrote preds [11000000:11500000]
Chunk 24: wrote preds [11500000:12000000]
Chunk 25: wrote preds [12000000:12500000]
Chunk 